In [ ]:
suppressPackageStartupMessages({
    library(ggpubr)
    library(patchwork)
    library(maplet)
    library(tidyverse)
    library(survival)
    library(glmnet)
    library(here)
})

# Load Data and Helpers

In [ ]:
# load tte helpers
source(here('helpers', 'prep_incident_time_to_event_df.r'))
source(here('helpers', 'summarize_time_to_event.r'))

In [ ]:
# load metadata and nmr data 
meta <- read.csv(here('data', 'meta_clin.csv'), row.names = 1)
assay <- read.csv(here('data', 'assay.csv'), row.names = 1) 
nmr_names <- assay %>% select(-StudyID) %>% names() # subset vector of nmr features 

## Load PRS

In [ ]:
# load prs and preprocess 
prs <- read.csv(here('data', 'prs.csv')) %>% 
    # remove '_hmPOS_GRCh38' from PGS column
    mutate(PGS = stringr::str_remove(PGS, '_hmPOS_GRCh38'))

prs <- distinct(prs) # obtain 1 PGS-specific score per patient

head(prs); dim(prs)

# Configuration:

In [ ]:
# cv glmnet helper
source(here('helpers', 'cv_glmnet_oof.r'))
source(here('helpers', 'fit_full_cox.r'))

# cv defaults
cv_defaults <- list(
  outer_nfolds   = 5,
  inner_nfolds   = 10,
  nlambda        = 100,
  seed           = 1,
  standardize    = FALSE,
  parallel_outer = TRUE,
  alpha = 0
)

common_blocks <- list(
  base = c("age_at_sample", "sex"),
  clin = c("BMI_interp", "SBP_interp"),
  prs  = c("SUM", "age_sum"),
  nmr  = nmr_names
)

common_scale_blocks <- list(
  base = c("age_at_sample"),
  clin = c("BMI_interp", "SBP_interp"),
  prs  = c("SUM", "age_sum"),
  nmr  = nmr_names
)

model_specs <- list(
  base           = c("base"),
  base_nmr       = c("base", "nmr"),
  base_prs       = c("base", "prs"),
  base_prs_nmr   = c("base", "prs", "nmr"),
  clin           = c("base", "clin"),
  clin_nmr       = c("base", "clin", "nmr"),
  clin_prs       = c("base", "clin", "prs"),
  clin_prs_nmr   = c("base", "clin", "prs", "nmr"),
  marker         = c("base", "clin", "marker"),
  marker_nmr     = c("base", "clin", "marker", "nmr"),
  marker_prs     = c("base", "clin", "marker", "prs"),
  marker_prs_nmr = c("base", "clin", "marker", "prs", "nmr")
)

In [ ]:
resolve_blocks <- function(block_names, block_map) {
  missing_blocks <- setdiff(block_names, names(block_map))
  if (length(missing_blocks) > 0) {
    stop("Missing block definitions: ", paste(missing_blocks, collapse = ", "))
  }
  unique(unlist(block_map[block_names], use.names = FALSE))
}

make_penalty_factor <- function(predictors, unpenalized = character()) {
  pf <- rep(1, length(predictors))
  names(pf) <- predictors
  pf[intersect(unpenalized, predictors)] <- 0.1
  pf
}

make_model_maps <- function(marker_predictors,
                            marker_scale = character()) {
  list(
    blocks = c(common_blocks, list(marker = marker_predictors)),
    scale_blocks = c(common_scale_blocks, list(marker = marker_scale))
  )
}

join_with_selected_prs <- function(meta_df, prs_df, prs_id) {
  prs_sub <- prs_df %>%
    filter(PGS == prs_id) %>%
    select(StudyID, SUM)

  meta_df %>%
    inner_join(prs_sub, by = "StudyID") %>%
    mutate(age_sum = age_at_sample * SUM)
}


get_required_vars <- function(maps,
                              specs = c("base_prs", "clin_prs", "marker_prs"),
                              extra_vars = c("time", "status"),
                              exclude_blocks = c("nmr")) {
  used_blocks <- unique(unlist(model_specs[specs], use.names = FALSE))
  used_blocks <- setdiff(used_blocks, exclude_blocks)

  vars_needed <- c(
    extra_vars,
    resolve_blocks(used_blocks, maps$blocks)
  )

  unique(vars_needed)
}


filter_complete_cases_for_specs <- function(df, maps,
                                            specs = c("base_prs", "clin_prs", "marker_prs"),
                                            extra_vars = c("time", "status"),
                                            exclude_blocks = c("nmr")) {
  vars_needed <- get_required_vars(
    maps = maps,
    specs = specs,
    extra_vars = extra_vars,
    exclude_blocks = exclude_blocks
  )

  df %>% filter(if_all(all_of(vars_needed), ~ !is.na(.)))
}

print_cv_summary <- function(results) {
  cat(sprintf("\nOOF C-index: %.4f (SE: %.4f)\n", results$c_index, results$se))
  cat(sprintf("95%% CI: [%.4f, %.4f]\n", results$ci_lower, results$ci_upper))
}

plot_cv_folds <- function(results) {
  n_folds <- length(results$fold_models)
  old_par <- par(no.readonly = TRUE)
  on.exit(par(old_par), add = TRUE)

  options(repr.plot.height = 2.5, repr.plot.width = 12)
  par(mfrow = c(1, n_folds))

  for (i in seq_len(n_folds)) {
    fold_model <- results$fold_models[[paste0("fold_", i)]]
    plot(fold_model, main = paste0("Fold\n", i))
  }
}

run_cv_spec <- function(df, spec_name, maps,
                        unpenalized = character(),
                        output_path = NULL,
                        plot_folds = FALSE,
                        model_engine = "glmnet",
                        fit_full = FALSE) {

  if (!spec_name %in% names(model_specs)) {
    stop("Unknown spec_name: ", spec_name)
  }

  block_names   <- model_specs[[spec_name]]
  predictors    <- resolve_blocks(block_names, maps$blocks)
  vars_to_scale <- resolve_blocks(block_names, maps$scale_blocks)

  # fail fast if requested predictors are not in df
  missing_predictors <- setdiff(c("time", "status", predictors), names(df))
  if (length(missing_predictors) > 0) {
    stop(
      sprintf(
        "Spec '%s' requested variables not found in df: %s",
        spec_name,
        paste(missing_predictors, collapse = ", ")
      )
    )
  }

  # also verify vars_to_scale are present
  missing_scale <- setdiff(vars_to_scale, names(df))
  if (length(missing_scale) > 0) {
    stop(
      sprintf(
        "Spec '%s' vars_to_scale not found in df: %s",
        spec_name,
        paste(missing_scale, collapse = ", ")
      )
    )
  }

  # restrict df to only the model-relevant columns
  df_model <- df %>%
    dplyr::select(dplyr::all_of(c("StudyID", "time", "status", predictors))) %>%
    dplyr::filter(dplyr::if_all(dplyr::all_of(c("time", "status", predictors)), ~ !is.na(.)))

  unpenalized <- intersect(unpenalized, predictors)
  penalty.factor <- make_penalty_factor(predictors, unpenalized)

  args <- c(
    list(
      df = df_model,
      predictors = predictors,
      vars_to_scale = vars_to_scale,
      penalty.factor = penalty.factor,
      model_engine = model_engine
    ),
    cv_defaults
  )
  results <- do.call(cv_glmnet_cox_oof, args)
  print_cv_summary(results)

  
  if (fit_full) {
    full_args <- c(
      list(
        df             = df_model,
        predictors     = predictors,
        vars_to_scale  = vars_to_scale,
        penalty.factor = penalty.factor,
        model_engine   = model_engine
      ),
      cv_defaults[setdiff(names(cv_defaults), c("outer_nfolds", "parallel_outer"))]
    )
    results$full_model <- do.call(fit_full_cox, full_args)
  }

  if (plot_folds && model_engine == "glmnet") {
    plot_cv_folds(results)
  }
  if (!is.null(output_path)) {
    saveRDS(results, output_path)
  }
  invisible(results)
}

inspect_spec <- function(df, spec_name, maps) {
  block_names <- model_specs[[spec_name]]
  predictors  <- resolve_blocks(block_names, maps$blocks)

  list(
    spec_name = spec_name,
    n_predictors = length(predictors),
    first_predictors = head(predictors, 10),
    missing_in_df = setdiff(predictors, names(df))
  )
}


# T2D

In [ ]:
# prepare data for time to event modeling (i.e. rename tte vars and exclude prevalent cases)
meta_use <- prep_incident_time_to_event_df(meta, status_col = 't2d_status', time_col = 't2d_time', time_filter = 't2d_time > 0')
summarize_time_to_event(meta_use)

In [ ]:
# join prs data to meta object
prs_id <- 'PGS002308'
meta_use <- join_with_selected_prs(meta_use, prs, prs_id)

In [ ]:
# define marker model maps 
t2d_maps <- make_model_maps(
  marker_predictors = c("M_a1c", "A1C_obs", "T_a1c"),
  marker_scale      = c("A1C_obs", "T_a1c")
)

In [ ]:
# filter for complete cases using t2d-specific covariates
meta_use <- filter_complete_cases_for_specs(
  df = meta_use,
  maps = t2d_maps,
  specs = c("base_prs", "clin_prs", "marker_prs")
)

# join nmr data 
meta_use_nmr <- meta_use %>% inner_join(assay, by = "StudyID")

cat("meta_use dimensions:", dim(meta_use), "\n")
cat("meta_use_nmr dimensions:", dim(meta_use_nmr), "\n")

In [ ]:
# report analytical set tte stats
t2d_analytical_tte <- summarize_time_to_event(meta_use) %>% mutate(cohort = 'KP', outcome = 'T2D')
t2d_analytical_tte 

## BASE

In [ ]:
# BASE
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base",
  maps = t2d_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/t2d_a1c_base_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


# BASE + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_prs",
  maps = t2d_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/t2d_a1c_base_prs_oof.rds"),
  fit_full = FALSE
)

# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_nmr",
  maps = t2d_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/t2d_a1c_base_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_prs_nmr",
  maps = t2d_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/t2d_a1c_base_prs_nmr_oof.rds"),
  fit_full = FALSE
)

# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


## CLIN

In [ ]:
# CLIN
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin",
  maps = t2d_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet", 
  output_path = here("outputs/main/t2d_a1c_clin_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_prs",
  maps = t2d_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/t2d_a1c_clin_prs_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_nmr",
  maps = t2d_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/t2d_a1c_clin_nmr_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_prs_nmr",
  maps = t2d_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/t2d_a1c_clin_prs_nmr_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


## MARKER

In [ ]:
# MARKER
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker",
  maps = t2d_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/t2d_a1c_marker_oof.rds"), 
  fit_full = TRUE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_prs",
  maps = t2d_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/t2d_a1c_marker_prs_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_nmr",
  maps = t2d_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/t2d_a1c_marker_nmr_oof.rds"), 
  fit_full = TRUE
)
rm(results) # clear model for memory

# MARKER + PRS + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_prs_nmr",
  maps = t2d_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/t2d_a1c_marker_prs_nmr_oof.rds"),
  fit_full = FALSE
)

rm(results) # clear model for memory


## OOF Absolute Risks 

### SOC

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved soc model 
soc_saved <- readRDS(here(
  "outputs/main",
  "t2d_a1c_marker_oof.rds"
))

# compute oof absolute risks (for various predictime times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved, # use the saved soc model 
  df = meta_use_nmr, # call the same df as the model 
  vars_to_scale = resolve_blocks( 
    model_specs[["marker"]], # use model specs for marker model (soc)
    t2d_maps$scale_blocks # use the same scale blocks as the model 
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/t2d_a1c_marker_fold_baseline_risk.rds")
)

### SOC+NMR

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved marker + nmr model 
soc_saved <- readRDS(here(
  "outputs/main",
  "t2d_a1c_marker_nmr_oof.rds"
))

# compute oof absolute risks (for various prediction times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved,
  df = meta_use_nmr,
  vars_to_scale = resolve_blocks(
    model_specs[["marker_nmr"]],
    t2d_maps$scale_blocks
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/t2d_a1c_marker_nmr_fold_baseline_risk.rds")
)

# CKD

In [ ]:
# prepare data for time to event modeling (i.e. rename tte vars and exclude prevalent cases)
meta_use <- prep_incident_time_to_event_df(meta, status_col = 'ckd_status', time_col = 'ckd_time', time_filter = 'ckd_time > 0')
summarize_time_to_event(meta_use)

In [ ]:
# join prs data to meta object
prs_id <- 'PGS000883'
meta_use <- join_with_selected_prs(meta_use, prs, prs_id)

In [ ]:
# define marker model maps 
ckd_maps <- make_model_maps(
  marker_predictors = c("M_egfr", "EGFR_obs", "T_egfr"),
  marker_scale      = c("EGFR_obs", "T_egfr")
)

In [ ]:
# filter for complete cases using ckd-specific covariates
meta_use <- filter_complete_cases_for_specs(
  df = meta_use,
  maps = ckd_maps,
  specs = c("base_prs", "clin_prs", "marker_prs")
)

# join nmr data 
meta_use_nmr <- meta_use %>% inner_join(assay, by = "StudyID")

cat("meta_use dimensions:", dim(meta_use), "\n")
cat("meta_use_nmr dimensions:", dim(meta_use_nmr), "\n")

In [ ]:
# report analytical set tte stats
ckd_analytical_tte <- summarize_time_to_event(meta_use) %>% mutate(cohort = 'KP', outcome = 'CKD')
ckd_analytical_tte 

## BASE

In [ ]:
# BASE
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base",
  maps = ckd_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/ckd_egfr_base_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


# BASE + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_prs",
  maps = ckd_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/ckd_egfr_base_prs_oof.rds"),
  fit_full = FALSE
)

# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_nmr",
  maps = ckd_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/ckd_egfr_base_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_prs_nmr",
  maps = ckd_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/ckd_egfr_base_prs_nmr_oof.rds"),
  fit_full = FALSE
)

# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


## CLIN

In [ ]:
# CLIN
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin",
  maps = ckd_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/ckd_egfr_clin_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_prs",
  maps = ckd_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/ckd_egfr_clin_prs_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_nmr",
  maps = ckd_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/ckd_egfr_clin_nmr_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_prs_nmr",
  maps = ckd_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/ckd_egfr_clin_prs_nmr_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


## MARKER

In [ ]:
# MARKER
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker",
  maps = ckd_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/ckd_egfr_marker_oof.rds"),
  fit_full = TRUE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_prs",
  maps = ckd_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/ckd_egfr_marker_prs_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_nmr",
  maps = ckd_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/ckd_egfr_marker_nmr_oof.rds"),
  fit_full = TRUE
)
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


# MARKER + PRS + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_prs_nmr",
  maps = ckd_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/ckd_egfr_marker_prs_nmr_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


## OOF Absolute Risks 

### SOC

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved soc model 
soc_saved <- readRDS(here(
  "outputs/main",
  "ckd_egfr_marker_oof.rds"
))

# compute oof absolute risks (for various predictime times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved, # use the saved soc model 
  df = meta_use_nmr, # call the same df as the model 
  vars_to_scale = resolve_blocks( 
    model_specs[["marker"]], # use model specs for marker model (soc)
    ckd_maps$scale_blocks # use the same scale blocks as the model 
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/ckd_egfr_marker_fold_baseline_risk.rds")
)

### SOC+NMR

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved marker + nmr model 
soc_saved <- readRDS(here(
  "outputs/main",
  "ckd_egfr_marker_nmr_oof.rds"
))

# compute oof absolute risks (for various prediction times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved,
  df = meta_use_nmr,
  vars_to_scale = resolve_blocks(
    model_specs[["marker_nmr"]],
    ckd_maps$scale_blocks
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/ckd_egfr_marker_nmr_fold_baseline_risk.rds")
)

# MASLD

In [ ]:
# prepare data for time to event modeling (i.e. rename tte vars and exclude prevalent cases)
meta_use <- prep_incident_time_to_event_df(meta, status_col = 'nafld_status', time_col = 'nafld_time', time_filter = 'nafld_time > 0')
summarize_time_to_event(meta_use)

In [ ]:
# join prs data to meta object
prs_id <- 'PGS002283'

meta_use <- join_with_selected_prs(meta_use, prs, prs_id)

In [ ]:
# define marker model maps 
nafld_maps <- make_model_maps(
  marker_predictors = c("M_fib4", "FIB4_obs", "T_fib4"),
  marker_scale      = c("FIB4_obs", "T_fib4")
)

In [ ]:
# filter for complete cases using masld-specific covariates
meta_use <- filter_complete_cases_for_specs(
  df = meta_use,
  maps = nafld_maps,
  specs = c("base_prs", "clin_prs", "marker_prs")
)

# join nmr data 
meta_use_nmr <- meta_use %>% inner_join(assay, by = "StudyID")

cat("meta_use dimensions:", dim(meta_use), "\n")
cat("meta_use_nmr dimensions:", dim(meta_use_nmr), "\n")

In [ ]:
# report analytical set tte stats
masld_analytical_tte <- summarize_time_to_event(meta_use) %>% mutate(cohort = 'KP', outcome = 'MASLD')
masld_analytical_tte 

## BASE

In [ ]:
## determine maps 
input_maps <- nafld_maps

# BASE
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/nafld_fib4_base_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


# BASE + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/nafld_fib4_base_prs_oof.rds"),
  fit_full = FALSE
)

# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


# BASE + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/nafld_fib4_base_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


# BASE + PRS + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/nafld_fib4_base_prs_nmr_oof.rds"),
  fit_full = FALSE
)

# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


## CLIN

In [ ]:
## determine maps 
input_maps <- nafld_maps

# CLIN
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/nafld_fib4_clin_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/nafld_fib4_clin_prs_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/nafld_fib4_clin_nmr_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/nafld_fib4_clin_prs_nmr_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


## MARKER

In [ ]:
## determine maps 
input_maps <- nafld_maps

# MARKER
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/nafld_fib4_marker_oof.rds"),
  fit_full = TRUE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/nafld_fib4_marker_prs_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/nafld_fib4_marker_nmr_oof.rds"),
  fit_full = TRUE
)
rm(results) # clear model for memory

# MARKER + PRS + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/nafld_fib4_marker_prs_nmr_oof.rds"),
  fit_full = FALSE
)

rm(results) # clear model for memory


## OOF Absolute Risks 

### SOC

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved soc model 
soc_saved <- readRDS(here(
  "outputs/main",
  "nafld_fib4_marker_oof.rds"
))

# compute oof absolute risks (for various predictime times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved, # use the saved soc model 
  df = meta_use_nmr, # call the same df as the model 
  vars_to_scale = resolve_blocks( 
    model_specs[["marker"]], # use model specs for marker model (soc)
    nafld_maps$scale_blocks # use the same scale blocks as the model 
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/nafld_fib4_marker_fold_baseline_risk.rds")
)

### SOC+NMR

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved marker + nmr model 
soc_saved <- readRDS(here(
  "outputs/main",
  "nafld_fib4_marker_nmr_oof.rds"
))

# compute oof absolute risks (for various prediction times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved,
  df = meta_use_nmr,
  vars_to_scale = resolve_blocks(
    model_specs[["marker_nmr"]],
    nafld_maps$scale_blocks
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/nafld_fib4_marker_nmr_fold_baseline_risk.rds")
)

# COPD

In [ ]:
# prepare data for time to event modeling (i.e. rename tte vars and exclude prevalent cases)
meta_use <- prep_incident_time_to_event_df(meta, status_col = 'copd_status', time_col = 'copd_time', time_filter = 'copd_time > 0')
summarize_time_to_event(meta_use)

In [ ]:
# define marker model maps 
copd_maps <- make_model_maps(
  marker_predictors = c("M_smoke", "smk_current", "smk_former", "M_ldl", "LDL_obs", "T_ldl"),
  marker_scale      = c("LDL_obs", "T_ldl")
)

In [ ]:
# filter for complete cases using copd-specific covariates
meta_use <- filter_complete_cases_for_specs(
  df = meta_use,
  maps = copd_maps,
  specs = c("base", "clin", "marker")
)

# join nmr data 
meta_use_nmr <- meta_use %>% inner_join(assay, by = "StudyID")

cat("meta_use dimensions:", dim(meta_use), "\n")
cat("meta_use_nmr dimensions:", dim(meta_use_nmr), "\n")

In [ ]:
# report analytical set tte stats
copd_analytical_tte <- summarize_time_to_event(meta_use) %>% mutate(cohort = 'KP', outcome = 'COPD')
copd_analytical_tte 

## BASE

In [ ]:
## determine maps 
input_maps <- copd_maps

# BASE
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/copd_smoking_ldl_base_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/copd_smoking_ldl_base_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## CLIN

In [ ]:
## determine maps 
input_maps <- copd_maps

# CLIN
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/copd_smoking_ldl_clin_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/copd_smoking_ldl_clin_nmr_oof.rds"),
  fit_full = FALSE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## MARKER

In [ ]:
## determine maps 
input_maps <- copd_maps

# MARKER
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/copd_smoking_ldl_marker_oof.rds"),
  fit_full = TRUE
)

results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/copd_smoking_ldl_marker_nmr_oof.rds"),
  fit_full = TRUE
)
rm(results) # clear model for memory

## OOF Absolute Risks 

### SOC

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved soc model 
soc_saved <- readRDS(here(
  "outputs/main",
  "copd_smoking_ldl_marker_oof.rds"
))

# compute oof absolute risks (for various predictime times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved, # use the saved soc model 
  df = meta_use_nmr, # call the same df as the model 
  vars_to_scale = resolve_blocks( 
    model_specs[["marker"]], # use model specs for marker model (soc)
    copd_maps$scale_blocks # use the same scale blocks as the model 
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/copd_smoking_ldl_marker_fold_baseline_risk.rds")
)

### SOC+NMR

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved marker + nmr model 
soc_saved <- readRDS(here(
  "outputs/main",
  "copd_smoking_ldl_marker_nmr_oof.rds"
))

# compute oof absolute risks (for various prediction times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved,
  df = meta_use_nmr,
  vars_to_scale = resolve_blocks(
    model_specs[["marker_nmr"]],
    copd_maps$scale_blocks
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/copd_smoking_ldl_marker_nmr_fold_baseline_risk.rds")
)

# CHD (Revasc)

In [ ]:
# prepare data for time to event modeling (i.e. rename tte vars and exclude prevalent cases)
meta_use <- prep_incident_time_to_event_df(meta, status_col = 'chd_minerva_status', time_col = 'chd_minerva_time', time_filter = 'chd_minerva_time > 0')
summarize_time_to_event(meta_use)

In [ ]:
# join prs data to meta object
prs_id <- 'PGS005039'
meta_use <- join_with_selected_prs(meta_use, prs, prs_id)

In [ ]:
# define marker model maps
# PREVENT ingredients not already included in base/clin:
#   - Lipids: total cholesterol + HDL, each with M/value/time blocks
#   - Renal function: eGFR with M/value/time block
#   - Smoking: survey response indicator + smoking-status dummies
#   - Treatments/comorbidity: BP meds, statin use, diabetes
chd_complete_maps <- make_model_maps(
  marker_predictors = c(
    # Total cholesterol block:
    "M_tot_choles",
    "TOT_CHOLES_obs",
    "T_tot_choles",
    # HDL cholesterol block:
    "M_hdl_choles",
    "HDL_CHOLES_obs",
    "T_hdl_choles",
    # Kidney function block:
    "M_egfr",
    "EGFR_obs",
    "T_egfr",
    # Smoking block:
    # M_smoke = whether smoking status was observed
    "M_smoke",
    "smk_current",
    "smk_former",
    # Medication / disease indicators:
    # bp_treatment = baseline antihypertensive treatment
    # statin_treatment = baseline statin treatment
    # diabetes_at_baseline = prevalent diabetes at baseline
    "bp_treatment",
    "statin_treatment",
    "diabetes_at_baseline"
  ),

  # Scale only continuous observed values and time-delay terms.
  # Do not scale binary indicators: M_*, smoking dummies, treatments, diabetes.
  marker_scale = c(
    "TOT_CHOLES_obs",
    "T_tot_choles",
    "HDL_CHOLES_obs",
    "T_hdl_choles",
    "EGFR_obs",
    "T_egfr"
  )
)

In [ ]:
# filter for complete cases using PREVENT-ingredient covariates
meta_use <- filter_complete_cases_for_specs(
  df = meta_use,
  maps = chd_complete_maps,
  specs = c("base_prs", "clin_prs", "marker_prs")
)

# join nmr data 
meta_use_nmr <- meta_use %>% inner_join(assay, by = "StudyID")

cat("meta_use dimensions:", dim(meta_use), "\n")
cat("meta_use_nmr dimensions:", dim(meta_use_nmr), "\n")

In [ ]:
# report analytical set tte stats
chd_analytical_tte <- summarize_time_to_event(meta_use) %>% mutate(cohort = 'KP', outcome = 'CHD')
chd_analytical_tte 

## BASE

In [ ]:
## determine maps 
input_maps <- chd_complete_maps

# BASE
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/chd_revasc_prevent_ingredients_base_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/chd_revasc_prevent_ingredients_base_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/chd_revasc_prevent_ingredients_base_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/chd_revasc_prevent_ingredients_base_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## CLIN

In [ ]:
## determine maps 
input_maps <- chd_complete_maps

# CLIN
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/chd_revasc_prevent_ingredients_clin_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/chd_revasc_prevent_ingredients_clin_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/chd_revasc_prevent_ingredients_clin_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/chd_revasc_prevent_ingredients_clin_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## MARKER

In [ ]:
## determine maps 
input_maps <- chd_complete_maps

# MARKER
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  fit_full = TRUE,
  output_path = here("outputs/main/chd_revasc_prevent_ingredients_marker_oof.rds"),
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  fit_full = FALSE,
  model_engine = "glmnet",
  output_path = here("outputs/main/chd_revasc_prevent_ingredients_marker_prs_oof.rds"),
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


# MARKER + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  fit_full = TRUE,
  output_path = here("outputs/main/chd_revasc_prevent_ingredients_marker_nmr_oof.rds"),
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  fit_full = FALSE,
  output_path = here("outputs/main/chd_revasc_prevent_ingredients_marker_prs_nmr_oof.rds"), 
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


## OOF Absolute Risks 

### SOC

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved soc model 
soc_saved <- readRDS(here(
  "outputs/main",
  "chd_revasc_prevent_ingredients_marker_oof.rds"
))

# compute oof absolute risks (for various predictime times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved, # use the saved soc model 
  df = meta_use_nmr, # call the same df as the model 
  vars_to_scale = resolve_blocks( 
    model_specs[["marker"]], # use model specs for marker model (soc)
    chd_complete_maps$scale_blocks # use the same scale blocks as the model 
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/chd_revasc_prevent_ingredients_marker_fold_baseline_risk.rds")
)

### SOC+NMR

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved marker + nmr model 
soc_saved <- readRDS(here(
  "outputs/main",
  "chd_revasc_prevent_ingredients_marker_nmr_oof.rds"
))

# compute oof absolute risks (for various prediction times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved,
  df = meta_use_nmr,
  vars_to_scale = resolve_blocks(
    model_specs[["marker_nmr"]],
    chd_complete_maps$scale_blocks
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/chd_revasc_prevent_ingredients_marker_nmr_fold_baseline_risk.rds")
)

# Heart Failure

In [ ]:
# prepare data for time to event modeling (i.e. rename tte vars and exclude prevalent cases)
meta_use <- prep_incident_time_to_event_df(meta, status_col = 'hf_cm_status', time_col = 'hf_cm_time', time_filter = 'hf_cm_time > 0')
summarize_time_to_event(meta_use)

In [ ]:
# join prs data to meta object
prs_id <- 'PGS005035'
meta_use <- join_with_selected_prs(meta_use, prs, prs_id)

In [ ]:
# filter for complete cases using PREVENT-ingredient covariates
meta_use <- filter_complete_cases_for_specs(
  df = meta_use,
  maps = chd_complete_maps,
  specs = c("base_prs", "clin_prs", "marker_prs")
)

# join nmr data 
meta_use_nmr <- meta_use %>% inner_join(assay, by = "StudyID")

cat("meta_use dimensions:", dim(meta_use), "\n")
cat("meta_use_nmr dimensions:", dim(meta_use_nmr), "\n")

In [ ]:
# report analytical set tte stats
hf_analytical_tte <- summarize_time_to_event(meta_use) %>% mutate(cohort = 'KP', outcome = 'Heart Failure')
hf_analytical_tte 

## BASE

In [ ]:
input_maps <- chd_complete_maps

# BASE
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/hf_cm_prevent_ingredients_base_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/hf_cm_prevent_ingredients_base_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/hf_cm_prevent_ingredients_base_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/hf_cm_prevent_ingredients_base_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## CLIN

In [ ]:
input_maps <- chd_complete_maps

# CLIN
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/hf_cm_prevent_ingredients_clin_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/hf_cm_prevent_ingredients_clin_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/hf_cm_prevent_ingredients_clin_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/hf_cm_prevent_ingredients_clin_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## MARKER

In [ ]:
input_maps <- chd_complete_maps

# MARKER
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/hf_cm_prevent_ingredients_marker_oof.rds"),
  fit_full = TRUE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/hf_cm_prevent_ingredients_marker_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


# MARKER + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/hf_cm_prevent_ingredients_marker_nmr_oof.rds"),
  fit_full = TRUE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS + NMR
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/hf_cm_prevent_ingredients_marker_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


## OOF Absolute Risks 

### SOC

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved soc model 
soc_saved <- readRDS(here(
  "outputs/main",
  "hf_cm_prevent_ingredients_marker_oof.rds"
))

# compute oof absolute risks (for various predictime times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved, # use the saved soc model 
  df = meta_use_nmr, # call the same df as the model 
  vars_to_scale = resolve_blocks( 
    model_specs[["marker"]], # use model specs for marker model (soc)
    chd_complete_maps$scale_blocks # use the same scale blocks as the model 
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/hf_cm_prevent_ingredients_marker_fold_baseline_risk.rds")
)

### SOC+NMR

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved marker + nmr model 
soc_saved <- readRDS(here(
  "outputs/main",
  "hf_cm_prevent_ingredients_marker_nmr_oof.rds"
))

# compute oof absolute risks (for various prediction times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved,
  df = meta_use_nmr,
  vars_to_scale = resolve_blocks(
    model_specs[["marker_nmr"]],
    chd_complete_maps$scale_blocks
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/hf_cm_prevent_ingredients_marker_nmr_fold_baseline_risk.rds")
)

# Acute MI

In [ ]:
# prepare data for time to event modeling (i.e. rename tte vars and exclude prevalent cases)
meta_use <- prep_incident_time_to_event_df(meta, status_col = 'acute_mi_status', time_col = 'acute_mi_time', time_filter = 'acute_mi_time > 0')
summarize_time_to_event(meta_use)

In [ ]:
# join prs data to meta object
prs_id <- 'PGS005039'

meta_use <- join_with_selected_prs(meta_use, prs, prs_id)

In [ ]:
# filter for complete cases using PREVENT-ingredient covariates
meta_use <- filter_complete_cases_for_specs(
  df = meta_use,
  maps = chd_complete_maps,
  specs = c("base_prs", "clin_prs", "marker_prs")
)

# join nmr data 
meta_use_nmr <- meta_use %>% inner_join(assay, by = "StudyID")

cat("meta_use dimensions:", dim(meta_use), "\n")
cat("meta_use_nmr dimensions:", dim(meta_use_nmr), "\n")

In [ ]:
# report analytical set tte stats
ami_analytical_tte <- summarize_time_to_event(meta_use) %>% mutate(cohort = 'KP', outcome = 'Acute MI')
ami_analytical_tte 

## BASE

In [ ]:
input_maps <- chd_complete_maps

# BASE
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/acute_mi_prevent_ingredients_base_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/acute_mi_prevent_ingredients_base_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/acute_mi_prevent_ingredients_base_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/acute_mi_prevent_ingredients_base_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## CLIN

In [ ]:
input_maps <- chd_complete_maps

# CLIN
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/acute_mi_prevent_ingredients_clin_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/acute_mi_prevent_ingredients_clin_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/acute_mi_prevent_ingredients_clin_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/acute_mi_prevent_ingredients_clin_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## MARKER

In [ ]:
input_maps <- chd_complete_maps

# MARKER
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/acute_mi_prevent_ingredients_marker_oof.rds"),
  fit_full = TRUE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/acute_mi_prevent_ingredients_marker_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/acute_mi_prevent_ingredients_marker_nmr_oof.rds"),
  fit_full = TRUE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/acute_mi_prevent_ingredients_marker_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


## OOF Absolute Risks 

### SOC

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved soc model 
soc_saved <- readRDS(here(
  "outputs/main",
  "acute_mi_prevent_ingredients_marker_oof.rds"
))

# compute oof absolute risks (for various predictime times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved, # use the saved soc model 
  df = meta_use_nmr, # call the same df as the model 
  vars_to_scale = resolve_blocks( 
    model_specs[["marker"]], # use model specs for marker model (soc)
    chd_complete_maps$scale_blocks # use the same scale blocks as the model 
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/acute_mi_prevent_ingredients_marker_fold_baseline_risk.rds")
)

### SOC+NMR

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved marker + nmr model 
soc_saved <- readRDS(here(
  "outputs/main",
  "acute_mi_prevent_ingredients_marker_nmr_oof.rds"
))

# compute oof absolute risks (for various prediction times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved,
  df = meta_use_nmr,
  vars_to_scale = resolve_blocks(
    model_specs[["marker_nmr"]],
    chd_complete_maps$scale_blocks
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/acute_mi_prevent_ingredients_marker_nmr_fold_baseline_risk.rds")
)

# TIA/Stroke

In [ ]:
# prepare data for time to event modeling (i.e. rename tte vars and exclude prevalent cases)
meta_use <- prep_incident_time_to_event_df(meta, status_col = 'tia_stroke_status', time_col = 'tia_stroke_time', time_filter = 'tia_stroke_time > 0')
summarize_time_to_event(meta_use)

In [ ]:
# join prs data to meta object
prs_id <- 'PGS005035'
meta_use <- join_with_selected_prs(meta_use, prs, prs_id)

In [ ]:
# filter for complete cases using PREVENT-ingredient covariates
meta_use <- filter_complete_cases_for_specs(
  df = meta_use,
  maps = chd_complete_maps,
  specs = c("base_prs", "clin_prs", "marker_prs")
)

# join nmr data 
meta_use_nmr <- meta_use %>% inner_join(assay, by = "StudyID")

cat("meta_use dimensions:", dim(meta_use), "\n")
cat("meta_use_nmr dimensions:", dim(meta_use_nmr), "\n")

In [ ]:
# report analytical set tte stats
tia_stroke_analytical_tte <- summarize_time_to_event(meta_use) %>% mutate(cohort = 'KP', outcome = 'TIA/Stroke')
tia_stroke_analytical_tte 

## BASE

In [ ]:
input_maps <- chd_complete_maps

# BASE
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/tia_stroke_prevent_ingredients_base_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/tia_stroke_prevent_ingredients_base_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/tia_stroke_prevent_ingredients_base_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/tia_stroke_prevent_ingredients_base_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## CLIN

In [ ]:
input_maps <- chd_complete_maps

# CLIN
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/tia_stroke_prevent_ingredients_clin_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/tia_stroke_prevent_ingredients_clin_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/tia_stroke_prevent_ingredients_clin_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/tia_stroke_prevent_ingredients_clin_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## MARKER

In [ ]:
input_maps <- chd_complete_maps

# MARKER
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/tia_stroke_prevent_ingredients_marker_oof.rds"),
  fit_full = TRUE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/tia_stroke_prevent_ingredients_marker_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


# MARKER + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/tia_stroke_prevent_ingredients_marker_nmr_oof.rds"),
  fit_full = TRUE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/tia_stroke_prevent_ingredients_marker_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


## OOF Absolute Risks 

### SOC

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved soc model 
soc_saved <- readRDS(here(
  "outputs/main",
  "tia_stroke_prevent_ingredients_marker_oof.rds"
))

# compute oof absolute risks (for various predictime times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved, # use the saved soc model 
  df = meta_use_nmr, # call the same df as the model 
  vars_to_scale = resolve_blocks( 
    model_specs[["marker"]], # use model specs for marker model (soc)
    chd_complete_maps$scale_blocks # use the same scale blocks as the model 
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/tia_stroke_prevent_ingredients_marker_fold_baseline_risk.rds")
)

### SOC+NMR

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved marker + nmr model 
soc_saved <- readRDS(here(
  "outputs/main",
  "tia_stroke_prevent_ingredients_marker_nmr_oof.rds"
))

# compute oof absolute risks (for various prediction times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved,
  df = meta_use_nmr,
  vars_to_scale = resolve_blocks(
    model_specs[["marker_nmr"]],
    chd_complete_maps$scale_blocks
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/tia_stroke_prevent_ingredients_marker_nmr_fold_baseline_risk.rds")
)

# Total CVD 

In [ ]:
# prepare data for time to event modeling (i.e. rename tte vars and exclude prevalent cases)
meta_use <- prep_incident_time_to_event_df(meta, status_col = 'total_cvd_status', time_col = 'total_cvd_time', time_filter = 'total_cvd_time > 0')
summarize_time_to_event(meta_use)

In [ ]:
# join prs data to meta object
prs_id <- 'PGS005035'
meta_use <- join_with_selected_prs(meta_use, prs, prs_id)

In [ ]:
# filter for complete cases using PREVENT-ingredient covariates
meta_use <- filter_complete_cases_for_specs(
  df = meta_use,
  maps = chd_complete_maps,
  specs = c("base_prs", "clin_prs", "marker_prs")
)

# join nmr data 
meta_use_nmr <- meta_use %>% inner_join(assay, by = "StudyID")

cat("meta_use dimensions:", dim(meta_use), "\n")
cat("meta_use_nmr dimensions:", dim(meta_use_nmr), "\n")

In [ ]:
# report analytical set tte stats
total_cvd_analytical_tte <- summarize_time_to_event(meta_use) %>% mutate(cohort = 'KP', outcome = 'Total CVD')
total_cvd_analytical_tte 

## BASE

In [ ]:
## determine maps 
input_maps <- chd_complete_maps

# BASE
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/total_cvd_prevent_ingredients_base_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/total_cvd_prevent_ingredients_base_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/total_cvd_prevent_ingredients_base_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/total_cvd_prevent_ingredients_base_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## CLIN

In [ ]:
## determine maps 
input_maps <- chd_complete_maps

# CLIN
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/total_cvd_prevent_ingredients_clin_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/total_cvd_prevent_ingredients_clin_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/total_cvd_prevent_ingredients_clin_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/total_cvd_prevent_ingredients_clin_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## MARKER

In [ ]:
## determine maps 
input_maps <- chd_complete_maps

# MARKER
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/total_cvd_prevent_ingredients_marker_oof.rds"),
  fit_full = TRUE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/total_cvd_prevent_ingredients_marker_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


# MARKER + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/total_cvd_prevent_ingredients_marker_nmr_oof.rds"),
  fit_full = TRUE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/total_cvd_prevent_ingredients_marker_prs_nmr_oof.rds"), 
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


## OOF Absolute Risks 

### SOC

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved soc model 
soc_saved <- readRDS(here(
  "outputs/main",
  "total_cvd_prevent_ingredients_marker_oof.rds"
))

# compute oof absolute risks (for various predictime times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved, # use the saved soc model 
  df = meta_use_nmr, # call the same df as the model 
  vars_to_scale = resolve_blocks( 
    model_specs[["marker"]], # use model specs for marker model (soc)
    chd_complete_maps$scale_blocks # use the same scale blocks as the model 
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/total_cvd_prevent_ingredients_marker_fold_baseline_risk.rds")
)

### SOC+NMR

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved marker + nmr model 
soc_saved <- readRDS(here(
  "outputs/main",
  "total_cvd_prevent_ingredients_marker_nmr_oof.rds"
))

# compute oof absolute risks (for various prediction times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved,
  df = meta_use_nmr,
  vars_to_scale = resolve_blocks(
    model_specs[["marker_nmr"]],
    chd_complete_maps$scale_blocks
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/total_cvd_prevent_ingredients_marker_nmr_fold_baseline_risk.rds")
)

# Afib


In [ ]:
# prepare data for time to event modeling (i.e. rename tte vars and exclude prevalent cases)
meta_use <- prep_incident_time_to_event_df(meta, status_col = 'afib_status', time_col = 'afib_time', time_filter = 'afib_time > 0')
summarize_time_to_event(meta_use)

In [ ]:
# join prs data to meta object
prs_id <- 'PGS005046'
meta_use <- join_with_selected_prs(meta_use, prs, prs_id)

In [ ]:
# define marker model maps
# CHARGE-AF ingredients not already included in base/clin:
#   - Anthropometrics: height (m), weight (kg)
#   - Blood pressure: diastolic (mmHg)
#   - Smoking: survey response indicator + current smoking status
#   - Treatments/comorbidity: BP meds, heart failure, MI, diabetes, race
afib_complete_maps <- make_model_maps(
  marker_predictors = c(
    "M_smoke_af",
    "smoking_current_af",
    "height_interp",
    "weight_interp",
    "DBP_interp",
    "bp_treatment",
    "hf_at_baseline",
    "mi_at_baseline",
    "diabetes_at_baseline",
    "race"
  ),

  # Scale only continuous observed values and time-delay terms.
  # Do not scale binary indicators: M_*, smoking dummies, treatments, diabetes.
  marker_scale = c(
    "height_interp",
    "weight_interp",
    "DBP_interp"
  )
)

In [ ]:
# filter for complete cases using CHARGE-AF-ingredient covariates
meta_use <- filter_complete_cases_for_specs(
  df = meta_use,
  maps = afib_complete_maps,
  specs = c("base_prs", "clin_prs", "marker_prs")
)

# join nmr data 
meta_use_nmr <- meta_use %>% inner_join(assay, by = "StudyID")

cat("meta_use dimensions:", dim(meta_use), "\n")
cat("meta_use_nmr dimensions:", dim(meta_use_nmr), "\n")

In [ ]:
# report analytical set tte stats
afib_analytical_tte <- summarize_time_to_event(meta_use) %>% mutate(cohort = 'KP', outcome = 'Afib')
afib_analytical_tte 

## BASE

In [ ]:
## determine maps 
input_maps <- afib_complete_maps

# BASE
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/afib_chargeaf_ingredients_base_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "base_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/afib_chargeaf_ingredients_base_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/afib_chargeaf_ingredients_base_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# BASE + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "base_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/afib_chargeaf_ingredients_base_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## CLIN

In [ ]:
## determine maps 
input_maps <- afib_complete_maps

# CLIN
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/afib_chargeaf_ingredients_clin_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "clin_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/afib_chargeaf_ingredients_clin_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/afib_chargeaf_ingredients_clin_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# CLIN + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "clin_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/afib_chargeaf_ingredients_clin_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

## MARKER

In [ ]:
## determine maps 
input_maps <- afib_complete_maps

# MARKER
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/afib_chargeaf_ingredients_marker_oof.rds"),
  fit_full = TRUE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS
results <- run_cv_spec(
  df = meta_use_nmr,
  spec_name = "marker_prs",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/afib_chargeaf_ingredients_marker_prs_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


# MARKER + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/afib_chargeaf_ingredients_marker_nmr_oof.rds"),
  fit_full = TRUE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory

# MARKER + PRS + NMR
results <- run_cv_spec(
   df = meta_use_nmr,
  spec_name = "marker_prs_nmr",
  maps = input_maps,
  unpenalized = c("age_at_sample", "sex"),
  model_engine = "glmnet",
  output_path = here("outputs/main/afib_chargeaf_ingredients_marker_prs_nmr_oof.rds"),
  fit_full = FALSE
)
# check summary of results
results$fold_info
summary(results$oof_predictions$lp)
rm(results) # clear model for memory


## OOF Absolute Risks 

### SOC

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved soc model 
soc_saved <- readRDS(here(
  "outputs/main",
  "afib_chargeaf_ingredients_marker_oof.rds"
))

# compute oof absolute risks (for various predictime times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved, # use the saved soc model 
  df = meta_use_nmr, # call the same df as the model 
  vars_to_scale = resolve_blocks( 
    model_specs[["marker"]], # use model specs for marker model (soc)
    afib_complete_maps$scale_blocks # use the same scale blocks as the model 
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/afib_chargeaf_ingredients_marker_fold_baseline_risk.rds")
)

### SOC+NMR

In [ ]:
# source oof absolute risk function 
source(here("helpers", "fold_specific_oof_absolute_risk.r"))

# load saved marker + nmr model 
soc_saved <- readRDS(here(
  "outputs/main",
  "afib_chargeaf_ingredients_marker_nmr_oof.rds"
))

# compute oof absolute risks (for various prediction times)
soc_oof_risk <- compute_fold_specific_oof_absolute_risk(
  results = soc_saved,
  df = meta_use_nmr,
  vars_to_scale = resolve_blocks(
    model_specs[["marker_nmr"]],
    afib_complete_maps$scale_blocks
  ),
  prediction_times = c(1, 3, 5)
)

# save the oof absolute risks 
saveRDS(
  soc_oof_risk,
  here("outputs/main/absolute_risks/afib_chargeaf_ingredients_marker_nmr_fold_baseline_risk.rds")
)

# Absolute C Index Analysis

In [ ]:
# Fig2A absolute C-index analysis from saved OOF RDS files (no refit)
library(dplyr)
library(ggplot2)
library(patchwork)
library(here)

source(here("helpers", "plot_oof_cindex_primary.r"))

V2_OOF_ROOT <- here("outputs/main")
V2_ROOT <- here("outputs")

dir.create(file.path(V2_ROOT, "c_index_summaries"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(V2_ROOT, "figures"), recursive = TRUE, showWarnings = FALSE)

FIG2A_ENDPOINTS <- list(
  t2d        = list(tag = "t2d_a1c",                              plot_label = "Type 2 Diabetes",                 table_label = "T2D"),
  ckd        = list(tag = "ckd_egfr",                             plot_label = "Chronic Kidney Disease",          table_label = "CKD"),
  masld      = list(tag = "nafld_fib4",                           plot_label = "MASLD (Liver Disease)",           table_label = "MASLD"),
  copd       = list(tag = "copd_smoking_ldl",                     plot_label = "Chronic Obstructive Pulmonary Disease", table_label = "COPD"),
  hf_cm      = list(tag = "hf_cm_prevent_ingredients",            plot_label = "Heart Failure",                   table_label = "Heart Failure"),
  acute_mi   = list(tag = "acute_mi_prevent_ingredients",         plot_label = "Acute MI",                        table_label = "Acute MI"),
  chd_revasc = list(tag = "chd_revasc_prevent_ingredients",       plot_label = "Coronary Heart Disease",          table_label = "CHD"),
  tia_stroke = list(tag = "tia_stroke_prevent_ingredients",       plot_label = "TIA/Stroke",                      table_label = "TIA/Stroke"),
  total_cvd  = list(tag = "total_cvd_prevent_ingredients",        plot_label = "Total CVD",                       table_label = "Total CVD"),
  afib       = list(tag = "afib_chargeaf_ingredients",            plot_label = "Atrial fibrillation",             table_label = "Afib")
)

FIG2A_PANEL_ORDER <- c(
  "t2d", "ckd", "masld",
  "hf_cm", "acute_mi", "chd_revasc",
  "tia_stroke", "total_cvd", "afib"
)

FIG2A_SPEC_MAP <- data.frame(
  spec  = c("base", "base_nmr", "clin", "clin_nmr", "marker", "marker_nmr"),
  model = c("base", "base", "clin", "clin", "marker", "marker"),
  type  = c("SOC", "NMR", "SOC", "NMR", "SOC", "NMR"),
  stringsAsFactors = FALSE
)

fig2a_oof_path <- function(tag, spec) {
  file.path(V2_OOF_ROOT, paste0(tag, "_", spec, "_oof.rds"))
}

fig2a_cindex_path <- function(tag) {
  file.path(
    V2_ROOT, "c_index_summaries",
    paste0("oof_updated_nmr_", tag, "_nofuture_cindex_ci.csv")
  )
}

write_fig2a_cindex_csv <- function(nm) {
  cfg <- FIG2A_ENDPOINTS[[nm]]
  if (is.null(cfg)) stop("Unknown Fig2A endpoint: ", nm)

  rows <- lapply(seq_len(nrow(FIG2A_SPEC_MAP)), function(i) {
    spec <- FIG2A_SPEC_MAP$spec[[i]]
    path <- fig2a_oof_path(cfg$tag, spec)
    if (!file.exists(path)) stop("Missing OOF RDS: ", path)

    res <- readRDS(path)
    data.frame(
      model    = FIG2A_SPEC_MAP$model[[i]],
      type     = FIG2A_SPEC_MAP$type[[i]],
      c_index  = res$c_index,
      ci_lower = res$ci_lower,
      ci_upper = res$ci_upper,
      stringsAsFactors = FALSE
    )
  })

  out <- dplyr::bind_rows(rows)
  path <- fig2a_cindex_path(cfg$tag)
  write.csv(out, path, row.names = FALSE)
  message("Wrote ", path)
  out
}

fig2a_cindex_list <- lapply(names(FIG2A_ENDPOINTS), write_fig2a_cindex_csv)
names(fig2a_cindex_list) <- names(FIG2A_ENDPOINTS)

fig2a_summary <- dplyr::bind_rows(lapply(names(fig2a_cindex_list), function(nm) {
  cfg <- FIG2A_ENDPOINTS[[nm]]
  fig2a_cindex_list[[nm]] %>%
    mutate(endpoint = nm, label = cfg$table_label, tag = cfg$tag, .before = model)
}))

write.csv(
  fig2a_summary,
  file.path(V2_ROOT, "c_index_summaries", "oof_updated_nmr_nofuture_cindex_summary.csv"),
  row.names = FALSE
)

make_fig2a_panel <- function(nm, xlim = c(0.5, 0.85)) {
  cfg <- FIG2A_ENDPOINTS[[nm]]
  plot_oof_cindex(
    df = fig2a_cindex_list[[nm]],
    title = cfg$plot_label,
    xlim = xlim,
    marker_label = "SOC",
    type_labels = c("SOC" = "No NMR", "NMR" = "NMR")
  )
}

fig2a_plots <- lapply(FIG2A_PANEL_ORDER, make_fig2a_panel)
names(fig2a_plots) <- FIG2A_PANEL_ORDER

options(repr.plot.height = 4, repr.plot.width = 7)

p_fig2a <-
  (fig2a_plots$t2d + theme(legend.position = "none") |
   fig2a_plots$ckd + theme(legend.position = "none") |
   fig2a_plots$masld + theme(legend.position = "none")) /
  (fig2a_plots$hf_cm + theme(legend.position = "none") |
   fig2a_plots$acute_mi + theme(legend.position = "none") |
   fig2a_plots$chd_revasc + theme(legend.position = "none")) /
  (fig2a_plots$tia_stroke + theme(legend.position = "none") |
   fig2a_plots$total_cvd + theme(legend.position = "bottom") |
   fig2a_plots$afib + theme(legend.position = "none"))

p_fig2a

ggsave(file.path(V2_ROOT, "figures", "FIG_2A.png"), plot = p_fig2a, width = 7, height = 4, dpi = 600)

# Delta C-index Analysis

Paired compareC tests from saved OOF `.rds` files (no refit), using the same six non-PRS Fig2A models.

In [ ]:
# Fig2B delta C-index setup from saved OOF RDS files (no refit)
library(dplyr)
library(ggplot2)
library(ggh4x)
library(here)

source(here("helpers", "delta_cindex_oof.r"))

V2_OOF_ROOT <- here("outputs/main")
V2_ROOT <- here("outputs")

dir.create(file.path(V2_ROOT, "c_index_summaries"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(V2_ROOT, "figures"), recursive = TRUE, showWarnings = FALSE)

# Fig2B omits COPD to match the primary manuscript figure.
if (!exists("FIG2A_ENDPOINTS")) {
  FIG2A_ENDPOINTS <- list(
    t2d        = list(tag = "t2d_a1c",                              plot_label = "Type 2 Diabetes",                 table_label = "T2D"),
    ckd        = list(tag = "ckd_egfr",                             plot_label = "Chronic Kidney Disease",          table_label = "CKD"),
    masld      = list(tag = "nafld_fib4",                           plot_label = "MASLD (Liver Disease)",           table_label = "MASLD"),
    copd       = list(tag = "copd_smoking_ldl",                     plot_label = "Chronic Obstructive Pulmonary Disease", table_label = "COPD"),
    hf_cm      = list(tag = "hf_cm_prevent_ingredients",            plot_label = "Heart Failure",                   table_label = "Heart Failure"),
    acute_mi   = list(tag = "acute_mi_prevent_ingredients",         plot_label = "Acute MI",                        table_label = "Acute MI"),
    chd_revasc = list(tag = "chd_revasc_prevent_ingredients",       plot_label = "Coronary Heart Disease",          table_label = "CHD"),
    tia_stroke = list(tag = "tia_stroke_prevent_ingredients",       plot_label = "TIA/Stroke",                      table_label = "TIA/Stroke"),
    total_cvd  = list(tag = "total_cvd_prevent_ingredients",        plot_label = "Total CVD",                       table_label = "Total CVD"),
    afib       = list(tag = "afib_chargeaf_ingredients",            plot_label = "Atrial fibrillation",             table_label = "Afib")
  )
}

DELTA_ENDPOINTS <- FIG2A_ENDPOINTS[c(
  "t2d", "ckd", "masld",
  "hf_cm", "acute_mi", "chd_revasc",
  "tia_stroke", "total_cvd", "afib"
)]

DELTA_SPEC_MAP <- c(
  base       = "base_SOC",
  base_nmr   = "base_NMR",
  clin       = "clin_SOC",
  clin_nmr   = "clin_NMR",
  marker     = "marker_SOC",
  marker_nmr = "marker_NMR"
)

DELTA_COMPARISONS <- data.frame(
  model_1 = c("base_SOC",   "clin_SOC",   "marker_SOC", "marker_SOC"),
  model_2 = c("base_NMR",   "clin_NMR",   "marker_NMR", "clin_NMR"),
  stringsAsFactors = FALSE
)

fig2b_oof_path <- function(tag, spec) {
  file.path(V2_OOF_ROOT, paste0(tag, "_", spec, "_oof.rds"))
}

fig2b_delta_path <- function(tag) {
  file.path(
    V2_ROOT, "c_index_summaries",
    paste0("oof_updated_nmr_", tag, "_nofuture_delta_cindex.csv")
  )
}

load_fig2b_oof_results <- function(tag) {
  results <- lapply(names(DELTA_SPEC_MAP), function(sp) {
    path <- fig2b_oof_path(tag, sp)
    if (!file.exists(path)) {
      stop("Missing OOF RDS: ", path)
    }
    strip_oof_result(readRDS(path))
  })
  names(results) <- unname(DELTA_SPEC_MAP)
  results
}

run_fig2b_delta <- function(nm,
                            skip_existing = TRUE,
                            parallel = FALSE,
                            workers = 2L) {
  cfg <- DELTA_ENDPOINTS[[nm]]
  if (is.null(cfg)) stop("Unknown delta endpoint: ", nm)

  path <- fig2b_delta_path(cfg$tag)
  if (skip_existing && file.exists(path)) {
    message("Skipping existing delta CSV: ", basename(path))
    return(
      read.csv(path, stringsAsFactors = FALSE) %>%
        mutate(
          endpoint = cfg$table_label,
          delta_endpoint = nm,
          tag = cfg$tag
        )
    )
  }

  message("Delta C: ", nm, " (", cfg$table_label, ")")
  results <- load_fig2b_oof_results(cfg$tag)
  delta <- run_delta_cindex_batch(
    results = results,
    comparisons = DELTA_COMPARISONS,
    strip_first = TRUE,
    parallel = parallel,
    workers = workers,
    parallel_transport = "rds",
    close_workers_on_exit = TRUE
  )
  delta$endpoint <- cfg$table_label
  delta$delta_endpoint <- nm
  delta$tag <- cfg$tag
  write.csv(delta, path, row.names = FALSE)
  message("Wrote ", path)

  rm(results)
  invisible(gc(verbose = FALSE))
  delta
}


In [ ]:
# Per-endpoint compareC. skip_existing = TRUE resumes from written CSVs.
delta_failed <- character()
for (nm in names(DELTA_ENDPOINTS)) {
  tryCatch(
    run_fig2b_delta(nm, skip_existing = FALSE, parallel = TRUE), # re-run all endpoitns (skip_existing = FALSE); parallel = TRUE (new) 
    error = function(e) {
      delta_failed <<- c(delta_failed, nm)
      warning("Failed delta C for ", nm, ": ", conditionMessage(e))
      NULL
    }
  )
}
if (length(delta_failed) > 0) {
  warning("Delta C failed for: ", paste(delta_failed, collapse = ", "))
} else {
  message("All Fig2B delta C endpoints finished (or were already on disk).")
}



In [ ]:
# Load per-disease delta CSVs and BH-adjust within boost (same as primary FIG_2B).
delta_list <- lapply(names(DELTA_ENDPOINTS), function(nm) {
  cfg <- DELTA_ENDPOINTS[[nm]]
  path <- fig2b_delta_path(cfg$tag)
  if (!file.exists(path)) {
    message("Missing delta CSV: ", basename(path))
    return(NULL)
  }
  read.csv(path, stringsAsFactors = FALSE) %>%
    mutate(
      endpoint = cfg$table_label,
      delta_endpoint = nm,
      tag = cfg$tag
    )
})
names(delta_list) <- names(DELTA_ENDPOINTS)

delta_all <- dplyr::bind_rows(delta_list)

delta_all <- delta_all %>%
  mutate(
    boost = case_when(
      model_1 == "base_SOC"   & model_2 == "base_NMR"   ~ "BASE",
      model_1 == "clin_SOC"   & model_2 == "clin_NMR"   ~ "CLIN",
      model_1 == "marker_SOC" & model_2 == "marker_NMR" ~ "SOC",
      model_1 == "marker_SOC" & model_2 == "clin_NMR"   ~ "CLIN+NMR vs SOC",
      TRUE ~ NA_character_
    )
  ) %>%
  filter(!is.na(boost)) %>%
  group_by(boost) %>%
  mutate(p_adj_BH = p.adjust(p_value, method = "BH")) %>%
  ungroup()

delta_all

write.csv(
  delta_all,
  file.path(V2_ROOT, "c_index_summaries", "oof_updated_nmr_nofuture_delta_cindex_summary.csv"),
  row.names = FALSE
)


In [ ]:
# revised format (same aesthetics as primary FIG_2B)
plot_delta_c_boost <- function(
    df,
    x_max = 0.25,
    x_by = 0.10,
    x_min_comparison = -0.02,
    annotation_pad = 0.025
) {

  endpoint_order <- c(
    "T2D", "CKD", "MASLD",
    "Heart Failure", "Acute MI", "CHD",
    "TIA/Stroke", "Total CVD", "Afib"
  )

  boost_order <- c(
    "BASE",
    "CLIN",
    "SOC",
    "CLIN+NMR vs SOC"
  )

  plot_df <- df %>%
    mutate(
      boost = case_when(
        model_1 == "base_SOC"   & model_2 == "base_NMR"   ~ "BASE",
        model_1 == "clin_SOC"   & model_2 == "clin_NMR"   ~ "CLIN",
        model_1 == "marker_SOC" & model_2 == "marker_NMR" ~ "SOC",
        model_1 == "marker_SOC" & model_2 == "clin_NMR"   ~ "CLIN+NMR vs SOC",
        TRUE ~ NA_character_
      ),
      boost = factor(boost, levels = boost_order),
      endpoint = factor(endpoint, levels = rev(endpoint_order))
    ) %>%
    filter(!is.na(boost))

  plot_df <- plot_df %>%
    mutate(
      sig_x = if_else(delta_c >= 0, ci_upper + 0.004, ci_lower - 0.004),
      sig_hjust = if_else(delta_c >= 0, 0, 1),
      sig_label = case_when(
        is.na(p_adj_BH)  ~ NA_character_,
        p_adj_BH >= 0.05 ~ "n.s.",
        p_adj_BH < 0.001 ~ "***",
        p_adj_BH < 0.01  ~ "**",
        TRUE             ~ "*"
      )
    )

  short_x_labels <- function(x) {
    vapply(
      x,
      function(z) {
        if (abs(z) < 1e-10) return("0")
        if (z < 0) return(paste0("-", sub("^0", "", sprintf("%.2f", abs(z)))))
        sub("^0", "", sprintf("%.2f", z))
      },
      character(1)
    )
  }

  main_breaks <- seq(0, floor(x_max / x_by) * x_by, by = x_by)

  ggplot(plot_df, aes(x = delta_c, y = endpoint)) +
    geom_col(width = 0.62, fill = "steelblue") +
    geom_errorbarh(
      aes(xmin = ci_lower, xmax = ci_upper),
      height = 0.18,
      linewidth = 0.45
    ) +
    geom_text(
      data = plot_df %>% filter(!is.na(sig_label)),
      aes(x = sig_x, label = sig_label, hjust = sig_hjust),
      size = 2.7,
      vjust = 0.7
    ) +
    geom_vline(xintercept = 0, linewidth = 0.25, colour = "black") +
    facet_wrap(~ boost, nrow = 1, scales = "free_x") +
    ggh4x::facetted_pos_scales(
      x = list(
        boost %in% c("BASE", "CLIN", "SOC") ~
          scale_x_continuous(
            limits = c(0, x_max + annotation_pad),
            breaks = main_breaks,
            labels = short_x_labels,
            expand = expansion(mult = c(0, 0))
          ),
        boost == "CLIN+NMR vs SOC" ~
          scale_x_continuous(
            limits = c(x_min_comparison, x_max + annotation_pad),
            breaks = seq(0, x_max, by = x_by),
            labels = short_x_labels,
            expand = expansion(mult = c(0, 0))
          )
      )
    ) +
    scale_y_discrete(drop = FALSE, expand = expansion(add = c(0.50, 0.55))) +
    labs(
      x = expression(Delta~"C-index (95% CI)"),
      y = NULL,
      title = "Incremental NMR Improvement"
    ) +
    theme_bw(base_size = 11) +
    theme(
      strip.background = element_rect(fill = "grey92", colour = NA),
      strip.text = element_text(face = "plain", size = 9.5, margin = margin(t = 4, r = 2, b = 4, l = 2)),
      panel.border = element_blank(),
      panel.grid.major.y = element_blank(),
      panel.grid.major.x = element_line(colour = "grey90", linewidth = 0.3),
      panel.grid.minor = element_blank(),
      panel.spacing.x = grid::unit(0.35, "cm"),
      axis.line.x = element_line(colour = "black", linewidth = 0.4),
      axis.ticks.x = element_line(colour = "black", linewidth = 0.35),
      axis.ticks.y = element_blank(),
      axis.text.x = element_text(size = 8, colour = "black", margin = margin(t = 2)),
      axis.text.y = element_text(size = 8.7, colour = "black", margin = margin(r = 3)),
      axis.title.x = element_text(size = 10.5, margin = margin(t = 5)),
      plot.title = element_text(hjust = 0.5, face = "plain", size = 12, margin = margin(b = 6)),
      plot.margin = margin(t = 3, r = 5, b = 2, l = 3)
    )
}


In [ ]:
options(repr.plot.height = 2.15, repr.plot.width = 6.3)

p_fig2b <- plot_delta_c_boost(
  df = delta_all,
  x_max = 0.25,
  x_by = 0.10,
  x_min_comparison = -0.03,
  annotation_pad = 0.025
)

p_fig2b
ggsave(file.path(V2_ROOT, "figures", "FIG_2B.png"), plot = p_fig2b, width = 6.3, height = 2.15, dpi = 300)

# COPD Supplement

In [ ]:
copd_cindex <- if (exists("fig2a_cindex_list") && !is.null(fig2a_cindex_list$copd)) {
  fig2a_cindex_list$copd
} else {
  write_fig2a_cindex_csv("copd")
}

options(repr.plot.width = 4, repr.plot.height = 1.75)

p_copd <- plot_oof_cindex(
  df = copd_cindex,
  title = "COPD",
  xlim = c(0.5, 0.85),
  marker_label = "SOC",
  type_labels = c("SOC" = "No NMR", "NMR" = "NMR")
) +
  theme(legend.position = "bottom")

p_copd

ggsave(
  file.path(V2_ROOT, "figures", "COPD.png"),
  plot = p_copd,
  width = 3.5,
  height = 1.75,
  dpi = 600
)


In [ ]:
copd_cfg <- FIG2A_ENDPOINTS$copd
copd_delta_results <- load_fig2b_oof_results(copd_cfg$tag)

delta_copd <- run_delta_cindex_batch(
  results = copd_delta_results,
  comparisons = DELTA_COMPARISONS,
  strip_first = TRUE,
  parallel = FALSE,
  lp_higher_risk = TRUE
) %>%
  mutate(
    endpoint = copd_cfg$table_label,
    tag = copd_cfg$tag,
    p_adj_BH = p.adjust(p_value, method = "BH")
  )

write.csv(
  delta_copd,
  fig2b_delta_path(copd_cfg$tag),
  row.names = FALSE
)
rm(copd_delta_results)

plot_delta_c_single <- function(df, title = NULL, x_max = 0.08, x_by = 0.02,
                                annotation_pad = 0.012) {
  boost_order <- c("BASE", "CLIN", "SOC", "CLIN+NMR vs SOC")
  plot_df <- df %>%
    mutate(
      boost = case_when(
        model_1 == "base_SOC"   & model_2 == "base_NMR"   ~ "BASE",
        model_1 == "clin_SOC"   & model_2 == "clin_NMR"   ~ "CLIN",
        model_1 == "marker_SOC" & model_2 == "marker_NMR" ~ "SOC",
        model_1 == "marker_SOC" & model_2 == "clin_NMR"   ~ "CLIN+NMR vs SOC",
        TRUE ~ NA_character_
      ),
      boost = factor(boost, levels = rev(boost_order)),
      sig_x = if_else(delta_c >= 0, ci_upper + 0.004, ci_lower - 0.004),
      sig_hjust = if_else(delta_c >= 0, 0, 1),
      sig_label = case_when(
        is.na(p_adj_BH)  ~ NA_character_,
        p_adj_BH >= 0.05 ~ "n.s.",
        p_adj_BH < 0.001 ~ "***",
        p_adj_BH < 0.01  ~ "**",
        TRUE             ~ "*"
      )
    ) %>%
    filter(!is.na(boost))

  short_x_labels <- function(x) {
    vapply(x, function(z) {
      if (is.na(z)) return("")
      if (abs(z) < 1e-10) return("0")
      if (z < 0) return(paste0("\u2212", sub("^0", "", sprintf("%.2f", abs(z)))))
      sub("^0", "", sprintf("%.2f", z))
    }, character(1))
  }

  x_min <- min(0, min(plot_df$ci_lower, na.rm = TRUE) - 0.004)

  ggplot(plot_df, aes(x = delta_c, y = boost)) +
    geom_col(width = 0.62, fill = "steelblue") +
    geom_errorbarh(aes(xmin = ci_lower, xmax = ci_upper), height = 0.18, linewidth = 0.45) +
    geom_text(
      data = filter(plot_df, !is.na(sig_label)),
      aes(x = sig_x, label = sig_label, hjust = sig_hjust),
      size = 2.7, vjust = 0.7
    ) +
    geom_vline(xintercept = 0, linewidth = 0.25, colour = "black") +
    scale_x_continuous(
      limits = c(x_min, x_max + annotation_pad),
      breaks = seq(0, floor(x_max / x_by) * x_by, by = x_by),
      labels = short_x_labels,
      expand = expansion(mult = c(0, 0))
    ) +
    scale_y_discrete(drop = FALSE, expand = expansion(add = c(0.50, 0.55))) +
    labs(x = expression(Delta~"C-index (95% CI)"), y = NULL, title = title) +
    theme_bw(base_size = 11) +
    theme(
      panel.border = element_blank(),
      panel.grid.major.y = element_blank(),
      panel.grid.major.x = element_line(colour = "grey90", linewidth = 0.3),
      panel.grid.minor = element_blank(),
      axis.line.x = element_line(colour = "black", linewidth = 0.4),
      axis.ticks.x = element_line(colour = "black", linewidth = 0.35),
      axis.ticks.y = element_blank(),
      axis.text.x = element_text(size = 8, colour = "black", margin = margin(t = 2)),
      axis.text.y = element_text(size = 8.7, colour = "black", margin = margin(r = 3)),
      axis.title.x = element_text(size = 10.5, margin = margin(t = 5)),
      plot.title = element_text(hjust = 0.5, face = "plain", size = 12, margin = margin(b = 6)),
      plot.margin = margin(t = 3, r = 5, b = 2, l = 3)
    )
}

options(repr.plot.height = 1.8, repr.plot.width = 4)
p_copd_delta <- plot_delta_c_single(delta_copd, title = "COPD")
p_copd_delta

ggsave(
  file.path(V2_ROOT, "figures", "COPD_DELTA.png"),
  plot = p_copd_delta,
  width = 3.5,
  height = 1.25,
  dpi = 600
)


# C-Index Stats

In [ ]:
# consolidate c-index stats across endpoint and model specs
# Six primary models (no PRS): BASE, CLIN, and SOC, each with and without NMR.
oof_dir <- here("outputs/main")

cindex_specs <- c(
  "T2D"           = "t2d_a1c_",
  "CKD"           = "ckd_egfr_",
  "MASLD"         = "nafld_fib4_",
  "COPD"          = "copd_smoking_ldl_",
  "Heart Failure" = "hf_cm_prevent_ingredients_",
  "Acute MI"      = "acute_mi_prevent_ingredients_",
  "CHD"           = "chd_revasc_prevent_ingredients_",
  "TIA/Stroke"    = "tia_stroke_prevent_ingredients_",
  "Total CVD"     = "total_cvd_prevent_ingredients_",
  "Afib"          = "afib_chargeaf_ingredients_"
)

cindex_files <- c(
  "BASE"      = "base_oof.rds",
  "BASE+NMR"  = "base_nmr_oof.rds",
  "CLIN"      = "clin_oof.rds",
  "CLIN+NMR"  = "clin_nmr_oof.rds",
  "SOC"       = "marker_oof.rds",
  "SOC+NMR"   = "marker_nmr_oof.rds"
)

fmt_cindex <- function(res) {
  sprintf("%.3f (%.3f, %.3f)", res$c_index, res$ci_lower, res$ci_upper)
}

cindex_table <- bind_rows(lapply(names(cindex_specs), function(endpoint) {
  vals <- vapply(cindex_files, function(suffix) {
    fmt_cindex(readRDS(file.path(oof_dir, paste0(cindex_specs[[endpoint]], suffix))))
  }, character(1))
  tibble::tibble(endpoint = endpoint, !!!as.list(vals))
})) %>%
  mutate(endpoint = factor(endpoint, levels = names(cindex_specs))) %>%
  arrange(endpoint)

cindex_table

cindex_dir <- here("outputs/tables")
dir.create(cindex_dir, showWarnings = FALSE, recursive = TRUE)
write.csv(cindex_table, file.path(cindex_dir, "CINDEX_STATS_PRIMARY.csv"), row.names = FALSE)

# TTE Stats

In [ ]:
# consolidate tte stats and store csv
fmt_int <- function(x) format(round(x), big.mark = ",", trim = TRUE)
fmt_events <- function(n, N) sprintf("%s (%.1f%%)", fmt_int(n), 100 * n / N)
fmt_mean_range <- function(mean, min, max) sprintf("%.2f (%.2f-%.2f)", mean, min, max)

endpoint_order <- c(
  "T2D", "CKD", "MASLD", "COPD", "CHD",
  "Heart Failure", "Acute MI", "TIA/Stroke", "Total CVD", "Afib"
)

kp_tte <- bind_rows(
  t2d_analytical_tte,
  ckd_analytical_tte,
  masld_analytical_tte,
  copd_analytical_tte,
  chd_analytical_tte,
  hf_analytical_tte,
  ami_analytical_tte,
  tia_stroke_analytical_tte,
  total_cvd_analytical_tte,
  afib_analytical_tte
)

tte_table <- kp_tte %>%
  mutate(arm = if_else(status == 1, "incident", "censored")) %>%
  select(outcome, arm, n, mean_time, min_time, max_time) %>%
  pivot_wider(
    names_from = arm,
    values_from = c(n, mean_time, min_time, max_time)
  ) %>%
  transmute(
    Endpoint = factor(outcome, levels = endpoint_order),
    `KP events, n (%)` = fmt_events(n_incident, n_censored + n_incident),
    `KP Censored, n` = fmt_int(n_censored),
    `KP Censored mean (min-max), y` = fmt_mean_range(mean_time_censored, min_time_censored, max_time_censored),
    `KP Incident, n` = fmt_int(n_incident),
    `KP Incident mean (min-max), y` = fmt_mean_range(mean_time_incident, min_time_incident, max_time_incident)
  ) %>%
  arrange(Endpoint)

tte_table

tte_dir <- here("outputs/tables")
dir.create(tte_dir, showWarnings = FALSE, recursive = TRUE)
write.csv(tte_table, file.path(tte_dir, "TTE_STATS.csv"), row.names = FALSE)